# Build & Deploy OGC Application Packages using the OGC-App-Pack GitHub Action

This tutorial describes how to use the [OGC App Pack Generator](https://github.com/MAAP-Project/ogc-app-pack-generator) GitHub Action to build and deploy OGC application packages to MAAP directly from your algorithm repository.

## What is the OGC App Pack Generator Action?

The OGC App Pack Generator is a GitHub Action that builds and deploys Open Geospatial Consortium (OGC) application packages to MAAP. When it runs, it:

- converts an input algorithm configuration YAML file into a [CWL](https://www.commonwl.org/) (Common Workflow Language) workflow file,
- validates the workflow for compliance with CWL and OGC best practices, and
- builds a Docker image from your `Dockerfile` and pushes it to your repository's GitHub Container Registry.

Optionally, it can also register the resulting application package with a MAAP OGC API endpoint.

## When would I use it?

Use this Action when you want to automate the build and deployment of an OGC-compliant application package as part of a CI/CD pipeline — for example, building and deploying your application package every time you push changes to your algorithm repository. This is an alternative to building an application package interactively through the Algorithm Catalog plugin (see [Register Algorithm as an OGC Application Package](./build_application_packages.ipynb)).

> **Note:** The Action **writes to your repository** by committing the generated CWL files and pushing a Docker image. Because of this, avoid running it on untrusted pull requests.

## Before you start

You will need:

- A GitHub repository containing your algorithm code
- An algorithm configuration YAML file
- A Dockerfile OR the `algorithm_container_url` param set in your algorithm configuration YAML file to an existing, public image

## Instructions

1. Create a workflow file 

  From the root of your repository:

    ```
    touch .github/workflows/my_workflow.yml
    ```

2. Add the Action to the workflow file 

  Copy the sample below into `my_workflow.yml`:

    ```yaml
    on:
      push:
        branches:
          - '**'
    jobs:
      build_app_pack:
        runs-on: ubuntu-latest

        permissions:
          contents: write
          packages: write

        steps:
          - name: Checkout repo content
            uses: actions/checkout@v6

          - name: Use OGC App Pack Generator Action
            uses: MAAP-Project/ogc-app-pack-generator@1.0.0
            with:
              # Specify action inputs
              algorithm-configuration-path: my_algo_repo/algorithm_config.yml
              dockerfile-path: my_algo_repo/Dockerfile
              deploy-app-pack: true
              app-pack-register-endpoint: https://api.maap-project.org/api/ogc/processes
            env:
              # MAAP token is required to deploy the process
              MAAP_TOKEN: ${{ secrets.MAAP_TOKEN }}
    ```

3. Update the inputs 

  Update the action inputs where necessary:

| Input | Required | Default | Description |
|:---|:---:|:---:|:---|
| `algorithm-configuration-path` | Yes | – | Path to your algorithm configuration YAML file. |
| `dockerfile-path` | Yes | – | Path to the `Dockerfile` used to build the image. |
| `cwl-workflow-dir` | No | `cwl_workflows` | Directory where the generated CWL workflow files are written. |
| `deploy-app-pack` | No | `false` | Whether to deploy the application package to a registry. |
| `app-pack-register-endpoint` | No | – | URL that receives the registration request. |
| `MAAP_TOKEN` | No | – | MAAP token used in the deployment request. Provide it as a repository secret. |

4. Add the `MAAP_TOKEN` secret (only needed if `deploy-app-pack` is `true`). 

  First, retrieve your MAAP token from the [MAAP token management page](https://console.maap-project.org/profile/tokens). 
  
  Then, in your repository, go to **Settings → Secrets and variables → Actions**, create a new repository secret named `MAAP_TOKEN`, and set its value to your MAAP token.

5. Push to your repository 

  On each push, the Action runs automatically: it generates and validates the CWL workflow, builds and pushes the Docker image, and — if `deploy-app-pack` is enabled — registers the application package with the specified MAAP endpoint. You can monitor the run under the **Actions** tab of your repository.

## Generating a workflow locally (optional)

If you want to generate and validate the CWL workflow without running the Action, clone the [repository](https://github.com/MAAP-Project/ogc-app-pack-generator) and run:

```
python build_cwl_workflow.py --config-file data/algorithm_config.yml
```

This produces `cwl_workflows/process.cwl`. Note that running locally only generates the CWL workflow — it does not build the Docker image.

To validate the workflow, first install the `cwltool` and `ap-validator` packages:

```
pip install cwltool
pip install ogc_ap_validator
```

Then run the validators:

```
cwltool --validate cwl_workflows/process.cwl    # CWL validation
ap-validator cwl_workflows/process.cwl          # OGC best-practices validation
```